In [ ]:
import random
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import Callback, ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix
from transformers import BertTokenizer
from tensorflow.keras.optimizers import Adam 
from transformers import TFBertForSequenceClassification, TFBertModel, TFAutoModelForCausalLM, TFGPT2LMHeadModel, AutoTokenizer, TFAutoModel
import tensorflow as tf
import tensorflow_addons as tfa


In [ ]:
# 1. Set Python hash seed (for consistency in hashing)
os.environ['PYTHONHASHSEED'] = '0'

# 2. Set seeds for Python, NumPy, and TensorFlow
seed = 42
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

# 3. (Optional) Configure TensorFlow for deterministic operations
# This helps eliminate non-deterministic GPU ops
os.environ['TF_DETERMINISTIC_OPS'] = '1'

In [ ]:
train_data = pd.read_csv('train_data_no_dup1.csv')
val_data = pd.read_csv('val_data_no_dup1.csv')
test_data = pd.read_csv('test_data_no_dup1.csv')

test_data2 = pd.read_csv('test_data_no_dup2.csv')

# Rename columns
# train_data.columns = ["NFR1", "NFR2", "conflict status"]
# val_data.columns = ["NFR1", "NFR2", "conflict status"]
test_data2.columns = ["NFR1", "NFR2", "conflict status"]

In [ ]:
train_data.shape

In [ ]:
val_data.shape

In [ ]:
test_data.shape

In [ ]:
tok = AutoTokenizer.from_pretrained('facebook/opt-125m')

In [ ]:
max_sample_length = max(train_data['NFR1'].apply(lambda x: len(x.split())))
print(f"Maximum sample length: {max_sample_length}")

max_sample_length = max(train_data['NFR2'].apply(lambda x: len(x.split())))
print(f"Maximum sample length: {max_sample_length}")

In [ ]:
label_words = {0: " Neutral", 1: " Conflict"}   # note leading space (important for GPT-2 BPE)

# tok = AutoTokenizer.from_pretrained(model_id)
tok.pad_token = tok.eos_token
tok.padding_side = "right"

# cache token-ids for label words (no special tokens)
LABEL_IDS = {y: tok(y, add_special_tokens=False)["input_ids"] for y in label_words.values()}

In [ ]:
def make_prompts(df):
    # df columns: NFR1, NFR2, and y in {0,1} (0=Neutral, 1=Conflict)
    return [
        f"You are a requirements analyst. Given two requirements, decide if they are in Conflict or Neutral.\n\n"
        f"Answer with one word only: Conflict or Neutral.\n\n"
        f"Requirement 1: {r1}\nRequirement 2: {r2}\n\nAnswer:"
        for r1, r2 in zip(df["NFR1"], df["NFR2"])
    ]

def build_io(prompts, y_list, max_length=190):
    input_ids_list, attn_mask_list, labels_list = [], [], []
    for prompt, y in zip(prompts, y_list):
        # encode prompt and label separately
        enc_p = tok(prompt, add_special_tokens=False)["input_ids"]
        lab_text = label_words[int(y)]
        lab_ids  = tok(lab_text, add_special_tokens=False)["input_ids"]

        # truncate prompt if needed so label fits
        room = max_length
        if len(enc_p) + len(lab_ids) > max_length:
            enc_p = enc_p[: max_length - len(lab_ids)]

        # concat prompt + label tokens (teacher forcing)
        ids = enc_p + lab_ids
        attn = [1] * len(ids)

        # pad to max_length
        pad_len = max_length - len(ids)
        if pad_len > 0:
            ids  += [tok.pad_token_id] * pad_len
            attn += [0] * pad_len

        # labels: -100 everywhere except on label token positions
        labels = [-100] * max_length
        start = len(enc_p)
        end   = start + len(lab_ids)
        labels[start:end] = ids[start:end]  # supervise only the label tokens

        input_ids_list.append(ids)
        attn_mask_list.append(attn)
        labels_list.append(labels)

    # to tf tensors
    input_ids   = tf.constant(input_ids_list, dtype=tf.int32)
    attention_m = tf.constant(attn_mask_list, dtype=tf.int32)
    labels      = tf.constant(labels_list,    dtype=tf.int32)
    return {"input_ids": input_ids, "attention_mask": attention_m, "labels": labels}


In [ ]:
# build your lists
train_prompts = make_prompts(train_data)
val_prompts   = make_prompts(val_data)

train_features = build_io(train_prompts, train_data["conflict status"].tolist())
val_features   = build_io(val_prompts,   val_data["conflict status"].tolist())

BATCH = 16

def make_ds(features, batch=BATCH, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((
        {"input_ids": features["input_ids"],
         "attention_mask": features["attention_mask"],
         "labels": features["labels"]},
        tf.zeros((features["input_ids"].shape[0],), dtype=tf.float32)  # dummy targets
    ))
    if shuffle:
        ds = ds.shuffle(1000, reshuffle_each_iteration=True)
    return ds.batch(batch).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(train_features, shuffle=True)
val_ds   = make_ds(val_features,   shuffle=False)


In [ ]:
# batch_size = 16
# train_dataset = train_dataset.shuffle(buffer_size=len(train_dataset), reshuffle_each_iteration=True).batch(batch_size).prefetch(tf.data.AUTOTUNE)
# val_dataset = val_dataset.batch(batch_size)
# test_dataset = test_dataset.batch(batch_size)

In [ ]:
# Load the pre-trained BERT model
lm = TFAutoModelForCausalLM.from_pretrained('facebook/opt-125m', pad_token_id=tok.pad_token_id)

In [ ]:
# for layer in lm.layers:
#     for sublayer in layer.submodules:
#         sublayer.trainable = False


In [ ]:
for layer in lm.layers:
    for sublayer in layer.submodules:
        print(sublayer.name, sublayer.trainable)


In [ ]:
lm.resize_token_embeddings(len(tok))   # make sure PAD is in vocab


In [ ]:
# Keras wrapper: call the HF model, take its .loss, and train with that
inp_ids  = tf.keras.Input(shape=(190,), dtype=tf.int32, name="input_ids")
inp_mask = tf.keras.Input(shape=(190,), dtype=tf.int32, name="attention_mask")
inp_lbls = tf.keras.Input(shape=(190,), dtype=tf.int32, name="labels")

out = lm(input_ids=inp_ids, attention_mask=inp_mask, labels=inp_lbls, return_dict=True)
loss = out.loss
logits = out.logits  # not used for loss externally, but handy if you want metrics


In [ ]:
model = tf.keras.Model(inputs=[inp_ids, inp_mask, inp_lbls], outputs=logits)
model.add_loss(loss)  # HF computed loss drives training
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5))  # good starting LR


In [ ]:
for layer in model.layers:
    print(layer.name, layer.trainable)


In [ ]:
checkpoint_filepath = 'best_model.keras'

In [ ]:
model_checkpoint_callback = ModelCheckpoint(
        filepath=checkpoint_filepath,
        save_weights_only=True,  # Set to True if you only want to save weights, False to save the entire model
        monitor='val_loss', 
        mode='min',  
        save_best_only=True, 
        verbose=1  # Log a message whenever a new checkpoint is saved
    )

In [ ]:
# Train the model
epochs = 10  # You can adjust this

history = model.fit(train_ds, validation_data=val_ds, epochs=epochs, callbacks=[model_checkpoint_callback])



In [ ]:
sns.set()
fig = plt.figure(0, (12, 4))

# ax = plt.subplot(1, 2, 1)
# sns.lineplot( history.history['f1_score'], label='train')
# sns.lineplot(history.history['val_f1_score'], label='valid')
# plt.title('Accuracy')
# plt.tight_layout()

ax = plt.subplot(1, 2, 2)
sns.lineplot(history.history['loss'], label='train')
sns.lineplot(history.history['val_loss'], label='valid')
plt.title('Loss')
plt.tight_layout()

plt.show()

In [ ]:
# Load weights from the saved model
model.load_weights(checkpoint_filepath)

In [ ]:
# test_labels= test_data['conflict status'].values

In [ ]:
def classify_pair_generative(nfr1, nfr2, max_new_tokens=2):
    prompt = (
        f"You are a requirements analyst. Given two requirements, decide if they are in Conflict or Neutral.\n\n"
        f"Answer with one word only: Conflict or Neutral.\n\n"
        f"Requirement 1: {nfr1}\n"
        f"Requirement 2: {nfr2}\n\n"
        "Answer:"
    )
    enc = tok(prompt, return_tensors="tf", padding=False, truncation=True, max_length=190)
    gen_ids = lm.generate(
        input_ids=enc["input_ids"],
        attention_mask=enc["attention_mask"],
        max_new_tokens=max_new_tokens,
        do_sample=False,            # greedy is fine for classification
        eos_token_id=tok.eos_token_id,
        pad_token_id=tok.pad_token_id
    )
    # Extract just the newly generated part
    new = gen_ids[:, enc["input_ids"].shape[1]:]
    text = tok.batch_decode(new.numpy(), skip_special_tokens=True)[0].strip().lower()
    # simple mapping (robust to punctuation)
    if "conflict" in text:
        return 1, text
    if "neutral" in text:
        return 0, text
    # fallback: compare token ids to label ids
    new_ids = new.numpy().tolist()[0]
    if new_ids[:len(LABEL_IDS[" Conflict"])] == LABEL_IDS[" Conflict"]:
        return 1, text
    if new_ids[:len(LABEL_IDS[" Neutral"])] == LABEL_IDS[" Neutral"]:
        return 0, text
    return None, text  # undecided

# example:
# pred, raw = classify_pair_generative("System must log all access", "System must never store logs")
# print(pred, raw)


In [ ]:
def evaluate_with_existing_fn(test_df, label_col="conflict status"):
    y_true = test_df[label_col].astype(int).to_numpy()
    preds, raws = [], []

    for r1, r2 in zip(test_df["NFR1"], test_df["NFR2"]):
        pred, raw = classify_pair_generative(r1, r2)  # <-- your function
        preds.append(pred)     # can be 0/1 or None
        raws.append(raw)       # raw generated text (for debugging)

    preds = np.array(preds, dtype=object)

    # Handle undecided (None) so metrics compute cleanly
    undecided = np.array([p is None for p in preds])
    if undecided.any():
        # simple policy: map undecided to majority class in test labels (change if you prefer 0 or 1)
        majority = 1 if np.mean(y_true) >= 0.5 else 0
        preds_filled = preds.copy()
        preds_filled[undecided] = majority
        y_pred = preds_filled.astype(int)
    else:
        y_pred = preds.astype(int)

    # Metrics (binary, no sklearn needed)
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())

    eps = 1e-9
    accuracy  = (tp + tn) / max(len(y_true), 1)
    precision = tp / max(tp + fp, 1) if (tp + fp) > 0 else 0.0
    recall    = tp / max(tp + fn, 1) if (tp + fn) > 0 else 0.0
    f1        = 2 * precision * recall / max(precision + recall, eps) if (precision + recall) > 0 else 0.0
    undecided_rate = float(undecided.mean())

    report = {
        "n_samples": int(len(y_true)),
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "undecided_rate": undecided_rate,
        "tp": tp, "tn": tn, "fp": fp, "fn": fn,
    }
    return report, y_pred, raws


In [ ]:
report, y_pred, raw_text = evaluate_with_existing_fn(test_data, label_col="conflict status")
print(report)


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
print(confusion_matrix(test_data["conflict status"], y_pred))
print(classification_report(test_data["conflict status"], y_pred, digits=4))

In [ ]:
def classify_pair_generative(nfr1, nfr2, max_new_tokens=2, shots=None):
    """
    Few-shot aware, but defaults to zero-shot if shots=None.
    `shots` can be:
      - a DataFrame with columns ['NFR1','NFR2','label'] or ['NFR1','NFR2','conflict status']
        where label can be 'neutral'/'conflict' or 0/1
      - a list of dicts: {'NFR1': ..., 'NFR2': ..., 'label': 'neutral'|'conflict'|0|1}
    """
    # --- few-shot block (optional) ---
    fewshot_text = ""
    if shots is not None:
        # Normalize to iterable of dicts
        if hasattr(shots, "iterrows"):
            # DataFrame
            cols = set(shots.columns)
            if "label" in cols:
                lab_col = "label"
            elif "conflict status" in cols:
                lab_col = "conflict status"
            else:
                raise ValueError("Few-shot DataFrame must have 'label' or 'conflict status' column.")
            items = (
                {"NFR1": r["NFR1"], "NFR2": r["NFR2"], "label": r[lab_col]}
                for _, r in shots.iterrows()
            )
        else:
            # Assume list of dicts
            items = shots

        def _to_label_text(v):
            s = str(v).strip().lower()
            if s in {"1", "conflict"}: return "conflict"
            if s in {"0", "neutral"}:  return "neutral"
            raise ValueError(f"Unexpected few-shot label: {v!r}")

        blocks = []
        for ex in items:
            lab = _to_label_text(ex["label"])
            blocks.append(
                "Example:\n"
                f"Requirement 1: {ex['NFR1']}\n"
                f"Requirement 2: {ex['NFR2']}\n"
                f"Answer: {lab}\n"
            )
        fewshot_text = "\n".join(blocks) + ("\n\n" if blocks else "")

    # --- prompt (preserves your original style) ---
    prompt = (
        "You are a requirements analyst. Given two requirements, decide if they are in Conflict or Neutral.\n\n"
        f"Answer with one word only: Conflict or Neutral.\n\n"
        f"{fewshot_text}"
        f"Requirement 1: {nfr1}\n"
        f"Requirement 2: {nfr2}\n\n"
        "Answer:"  # keep this last so the next token is the label
    )

    enc = tok(prompt, return_tensors="tf", padding=False, truncation=True, max_length=190)
    gen_ids = lm.generate(
        input_ids=enc["input_ids"],
        attention_mask=enc["attention_mask"],
        max_new_tokens=max_new_tokens,
        do_sample=False,            # greedy is fine for classification
        eos_token_id=tok.eos_token_id,
        pad_token_id=tok.pad_token_id
    )
    # Extract just the newly generated part
    new = gen_ids[:, enc["input_ids"].shape[1]:]
    text = tok.batch_decode(new.numpy(), skip_special_tokens=True)[0].strip().lower()

    # simple mapping (robust to punctuation)
    if "conflict" in text:
        return 1, text
    if "neutral" in text:
        return 0, text

    # fallback: compare token ids to label ids, if you already have LABEL_IDS defined
    try:
        new_ids = new.numpy().tolist()[0]
        if new_ids[:len(LABEL_IDS[" Conflict"])] == LABEL_IDS[" Conflict"]:
            return 1, text
        if new_ids[:len(LABEL_IDS[" Neutral"])] == LABEL_IDS[" Neutral"]:
            return 0, text
    except Exception:
        pass

    return None, text  # undecided


def evaluate_with_existing_fn(test_df, label_col="conflict status", shots=None):
    """
    Added `shots` passthrough; zero-shot if shots=None.
    """
    y_true = test_df[label_col].astype(int).to_numpy()
    preds, raws = [], []

    for r1, r2 in zip(test_df["NFR1"], test_df["NFR2"]):
        pred, raw = classify_pair_generative(r1, r2, shots=shots)  # <-- now few-shot aware
        preds.append(pred)     # can be 0/1 or None
        raws.append(raw)       # raw generated text (for debugging)

    preds = np.array(preds, dtype=object)

    # Handle undecided (None) so metrics compute cleanly
    undecided = np.array([p is None for p in preds])
    if undecided.any():
        # simple policy: map undecided to majority class in test labels (change if you prefer 0 or 1)
        majority = 1 if np.mean(y_true) >= 0.5 else 0
        preds_filled = preds.copy()
        preds_filled[undecided] = majority
        y_pred = preds_filled.astype(int)
    else:
        y_pred = preds.astype(int)

    # Metrics (binary, no sklearn needed)
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())

    eps = 1e-9
    accuracy  = (tp + tn) / max(len(y_true), 1)
    precision = tp / max(tp + fp, 1) if (tp + fp) > 0 else 0.0
    recall    = tp / max(tp + fn, 1) if (tp + fn) > 0 else 0.0
    f1        = 2 * precision * recall / max(precision + recall, eps) if (precision + recall) > 0 else 0.0
    undecided_rate = float(undecided.mean())

    report = {
        "n_samples": int(len(y_true)),
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "undecided_rate": undecided_rate,
        "tp": tp, "tn": tn, "fp": fp, "fn": fn,
    }
    return report, y_pred, raws


In [ ]:
report, y_pred, raws = evaluate_with_existing_fn(test_data, label_col="conflict status")


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
print(confusion_matrix(test_data["conflict status"], y_pred))
print(classification_report(test_data["conflict status"], y_pred, digits=4))

In [ ]:
def _normalize_label_to_text(v):
    s = str(v).strip().lower()
    if s in {"1", "conflict"}: return "conflict"
    if s in {"0", "neutral"}:  return "neutral"
    raise ValueError(f"Unexpected label: {v!r}")

def sample_balanced_shots(df, k_per_class=1, seed=None):
    """
    df must have columns: NFR1, NFR2, and either 'label' or 'conflict status'.
    Returns a tiny balanced DF with columns: NFR1, NFR2, label ('neutral'/'conflict').
    """
    if "label" in df.columns:
        lab_col = "label"
    elif "conflict status" in df.columns:
        lab_col = "conflict status"
    else:
        raise ValueError("shots source must have 'label' or 'conflict status' column")

    tmp = df[["NFR1", "NFR2", lab_col]].dropna().copy()
    tmp["label"] = tmp[lab_col].map(_normalize_label_to_text)
    tmp = tmp[["NFR1", "NFR2", "label"]]

    # if any class has < k, enable replacement just for that class
    cnt = tmp["label"].value_counts()
    replace_flag = cnt.min() < k_per_class

    shots = (
        tmp.groupby("label", group_keys=False)
           .sample(n=k_per_class, replace=replace_flag, random_state=seed)
           .reset_index(drop=True)
    )
    return shots

def evaluate_with_existing_fn(
    test_df,
    label_col="conflict status",
    shots_source=None,      
    k_per_class=1,         
    base_seed=None          
):
    """
    If shots_source is provided, we re-sample fresh few-shots for EVERY test sample.
    If shots_source is None, we run zero-shot (exactly your old behavior).
    """
    y_true = test_df[label_col].astype(int).to_numpy()
    preds, raws = [], []

    for idx, (r1, r2) in enumerate(zip(test_df["NFR1"], test_df["NFR2"])):
        # draw new shots for THIS test example (or None for zero-shot)
        if shots_source is not None:
            # make per-iteration seeds reproducible-but-changing if base_seed is given
            seed = None if base_seed is None else (int(base_seed) + idx)
            fewshots = sample_balanced_shots(shots_source, k_per_class=k_per_class, seed=seed)
        else:
            fewshots = None  # zero-shot

        pred, raw = classify_pair_generative(r1, r2, shots=fewshots)  # uses your existing function
        preds.append(pred)   # 0/1 or None
        raws.append(raw)

    preds = np.array(preds, dtype=object)

    # Handle undecided (None) so metrics compute cleanly
    undecided = np.array([p is None for p in preds])
    if undecided.any():
        majority = 1 if np.mean(y_true) >= 0.5 else 0
        preds_filled = preds.copy()
        preds_filled[undecided] = majority
        y_pred = preds_filled.astype(int)
    else:
        y_pred = preds.astype(int)

    # Metrics (binary)
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())

    eps = 1e-9
    accuracy  = (tp + tn) / max(len(y_true), 1)
    precision = tp / max(tp + fp, 1) if (tp + fp) > 0 else 0.0
    recall    = tp / max(tp + fn, 1) if (tp + fn) > 0 else 0.0
    f1        = 2 * precision * recall / max(precision + recall, eps) if (precision + recall) > 0 else 0.0
    undecided_rate = float(undecided.mean())

    report = {
        "n_samples": int(len(y_true)),
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "undecided_rate": undecided_rate,
        "tp": tp, "tn": tn, "fp": fp, "fn": fn,
    }
    return report, y_pred, raws


In [ ]:
report, y_pred, raws = evaluate_with_existing_fn(test_data, label_col="conflict status")
print(confusion_matrix(test_data["conflict status"], y_pred))
print(classification_report(test_data["conflict status"], y_pred, digits=4))

In [ ]:
# use your training split as the source to draw examples from
shots_source = train_data  # must have NFR1, NFR2, and 'conflict status' (or 'label')

report, y_pred, raws = evaluate_with_existing_fn(
    test_df=test_data,
    label_col="conflict status",
    shots_source=shots_source,
    k_per_class=1,          # 1 shot per class per test item
    base_seed=None          # None => different (non-deterministic) each iteration
)
print(confusion_matrix(test_data["conflict status"], y_pred))
print(classification_report(test_data["conflict status"], y_pred, digits=4))

In [ ]:
# use your training split as the source to draw examples from
shots_source = train_data  # must have NFR1, NFR2, and 'conflict status' (or 'label')

report, y_pred, raws = evaluate_with_existing_fn(
    test_df=test_data,
    label_col="conflict status",
    shots_source=shots_source,
    k_per_class=2,          # 1 shot per class per test item
    base_seed=None          # None => different (non-deterministic) each iteration
)
print(confusion_matrix(test_data["conflict status"], y_pred))
print(classification_report(test_data["conflict status"], y_pred, digits=4))

In [ ]:
report, y_pred, raws = evaluate_with_existing_fn(test_data2, label_col="conflict status")
print(confusion_matrix(test_data2["conflict status"], y_pred))
print(classification_report(test_data2["conflict status"], y_pred, digits=4))